# Extract Custom Fields from Your File

This notebook demonstrates how to use analyzers to extract custom fields from your input files.

Source: https://github.com/Azure-Samples/azure-ai-content-understanding-python.git
further expanded for training purposes.

In [ ]:
import logging
import json
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import uuid # for generating unique analyzer IDs

## Analyzer Templates

Below is a collection of analyzer templates designed to extract fields from various input file types.

These templates are highly customizable, allowing you to modify them to suit your specific needs. For additional verified templates from Microsoft, please visit [here](https://github.com/Azure-Samples/azure-ai-content-understanding-python/tree/fb251ace83a98c9c82b9871f217c88d73e676bb7/analyzer_templates).

In [72]:
analyzer_template_folder = "analyzer_templates_NEW"
data_folder = "assets"

extraction_templates = {
    # Extract fields from invoices (no grounding sources or confidence scores).
    "invoice": (f'{analyzer_template_folder}/invoice.json', f'{data_folder}/docs/invoice.pdf'),

    # Extract fields from invoices, including grounding sources and confidence scores (optional add-on).
    "invoice_field_source": (f'{analyzer_template_folder}/invoice_field_source.json', f'{data_folder}/docs/invoice.pdf'),

    # Extract insights from call recordings (e.g., summary, topics, mentioned companies, and people).
    "call_recording": (f'{analyzer_template_folder}/call_recording_analytics.json', f'{data_folder}/audio/callCenterRecording.mp3'),

    # Extract summary and sentiment from conversation audio (e.g., customer service calls).
    "conversation_audio": (f'{analyzer_template_folder}/conversational_audio_analytics.json', f'{data_folder}/audio/callCenterRecording.mp3'),

    # Extract descriptions and sentiment analysis from marketing videos.
    "marketing_video": (f'{analyzer_template_folder}/marketing_video.json', f'{data_folder}/video/FlightSimulator.mp4'),
}

## Create Azure AI Content Understanding Client

> The [AzureContentUnderstandingClient](../python/content_understanding_client.py) is a utility class containing functions to interact with the Content Understanding API. Before the official release of the Content Understanding SDK, it can be regarded as a lightweight SDK.


In [58]:
load_dotenv(override=True, dotenv_path=find_dotenv())

True

In [59]:
logging.basicConfig(level=logging.INFO)

In [60]:
AZURE_AI_ENDPOINT = os.getenv("AZURE_CU_ENDPOINT_NEW")
AZURE_AI_API_VERSION =  os.getenv("AZURE_CU_API_VERSION_NEW", "2025-05-01-preview")
print(f"Current Azure Content Understanding endpoint: {AZURE_AI_ENDPOINT}")
print(f"Current Azure Content Understanding API version: {AZURE_AI_API_VERSION}")

Current Azure Content Understanding endpoint: https://epaifhub6672084982.cognitiveservices.azure.com/
Current Azure Content Understanding API version: 2025-05-01-preview


In [61]:
# only if necessary, add the parent directory to the path to use shared modules
# parent_dir = Path(Path.cwd()).parent
# sys.path.append(str(parent_dir))

# import the utility class AzureContentUnderstandingClient, which is a wrapper around the Azure Content Understanding REST API client
from python.content_understanding_client_NEW import AzureContentUnderstandingClient

In [62]:
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

INFO:azure.identity._credentials.environment:No environment configuration found.
INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use IMDS


In [63]:
client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    # x_ms_useragent="azure-ai-content-understanding-python/field_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.23.0 Python/3.13.4 (Windows-11-10.0.26100-SP0)'
No body was attached to the request
INFO:azure.identity._credentials.chained:DefaultAzureCredential acquired a token from AzureCliCredential


## Create Analyzer from the Template

Specify the analyzer template you want to use and provide a name for the analyzer to be created based on the template.

In [73]:
#extract keys from extraction_templates
analyzer_names = list(extraction_templates.keys())
print(f"Available analyzers: {analyzer_names}")

Available analyzers: ['invoice', 'invoice_field_source', 'call_recording', 'conversation_audio', 'marketing_video']


In [84]:
ANALYZER_TEMPLATE = "invoice_field_source"

(analyzer_template_path, analyzer_sample_file_path) = extraction_templates[ANALYZER_TEMPLATE]
print(f"Using analyzer template: {analyzer_template_path}")
print(f"Using sample file: {analyzer_sample_file_path}")

Using analyzer template: analyzer_templates_NEW/invoice_field_source.json
Using sample file: assets/docs/invoice.pdf


In [85]:
CUSTOM_ANALYZER_ID = "field-extraction-sample-" + str(uuid.uuid4())
response = client.begin_create_analyzer(CUSTOM_ANALYZER_ID, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client_NEW:Analyzer field-extraction-sample-e6edf977-91d1-43a9-8346-9b081c3c0a3c create request accepted.
INFO:python.content_understanding_client_NEW:Request 7d2ddfaf-c6e2-4002-b6ce-e3df65d77c5b in progress ...
INFO:python.content_understanding_client_NEW:Request result is ready after 2.46 seconds.


{
  "id": "7d2ddfaf-c6e2-4002-b6ce-e3df65d77c5b",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-e6edf977-91d1-43a9-8346-9b081c3c0a3c",
    "description": "Sample invoice analyzer",
    "createdAt": "2025-06-16T09:31:59Z",
    "lastModifiedAt": "2025-06-16T09:32:00Z",
    "baseAnalyzerId": "prebuilt-documentAnalyzer",
    "config": {
      "returnDetails": true,
      "enableOcr": true,
      "enableLayout": true,
      "enableFormula": false,
      "disableContentFiltering": false,
      "tableFormat": "html",
      "estimateFieldSourceAndConfidence": true
    },
    "fieldSchema": {
      "fields": {
        "VendorName": {
          "type": "string",
          "method": "extract",
          "description": "Vendor issuing the invoice"
        },
        "Items": {
          "type": "array",
          "method": "extract",
          "items": {
            "type": "object",
            "properties": {
              "Description": {
                "typ

## Extract Fields Using the Analyzer

After the analyzer is successfully created, we can use it to analyze our input files.

In [86]:
response = client.begin_analyze(CUSTOM_ANALYZER_ID, file_location=analyzer_sample_file_path)
result_json = client.poll_result(response, timeout_seconds=240)

print(json.dumps(result_json, indent=2))

INFO:python.content_understanding_client_NEW:Analyzing file assets/docs/invoice.pdf with analyzer: field-extraction-sample-e6edf977-91d1-43a9-8346-9b081c3c0a3c
INFO:python.content_understanding_client_NEW:Request 72ab6dc7-a5dd-4fd8-a6e8-9ddcc17cac25 in progress ...
INFO:python.content_understanding_client_NEW:Request 72ab6dc7-a5dd-4fd8-a6e8-9ddcc17cac25 in progress ...
INFO:python.content_understanding_client_NEW:Request result is ready after 5.06 seconds.


{
  "id": "72ab6dc7-a5dd-4fd8-a6e8-9ddcc17cac25",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-e6edf977-91d1-43a9-8346-9b081c3c0a3c",
    "apiVersion": "2025-05-01-preview",
    "createdAt": "2025-06-16T09:32:03Z",
    "warnings": [],
    "contents": [
      {
        "markdown": "CONTOSO LTD.\n\n\n# INVOICE\n\nContoso Headquarters\n123 456th St\nNew York, NY, 10001\n\nINVOICE: INV-100\n\nINVOICE DATE: 11/15/2019\n\nDUE DATE: 12/15/2019\n\nCUSTOMER NAME: MICROSOFT CORPORATION\n\nSERVICE PERIOD: 10/14/2019 - 11/14/2019\n\nCUSTOMER ID: CID-12345\n\nMicrosoft Corp\n123 Other St,\nRedmond WA, 98052\n\nBILL TO:\n\nMicrosoft Finance\n\n123 Bill St,\n\nRedmond WA, 98052\n\nSHIP TO:\n\nMicrosoft Delivery\n\n123 Ship St,\n\nRedmond WA, 98052\n\nSERVICE ADDRESS:\nMicrosoft Services\n123 Service St,\nRedmond WA, 98052\n\n\n<table>\n<tr>\n<th>SALESPERSON</th>\n<th>P.O. NUMBER</th>\n<th>REQUISITIONER</th>\n<th>SHIPPED VIA</th>\n<th>F.O.B. POINT</th>\n<th>TERMS</

In [87]:
# save the result to a json file
output_dir = "results/field_extraction"
Path(output_dir).mkdir(parents=True, exist_ok=True)
file_name = Path(analyzer_sample_file_path).name
output_file = f"{output_dir}/{ANALYZER_TEMPLATE}_{file_name}.json"

with open(output_file, "w") as f:
    json.dump(result_json, f, indent=2)

## Clean Up
Optionally, delete the sample analyzer from your resource. In typical usage scenarios, you would analyze multiple files using the same analyzer.

In [88]:
client.delete_analyzer(CUSTOM_ANALYZER_ID)

INFO:python.content_understanding_client_NEW:Analyzer field-extraction-sample-e6edf977-91d1-43a9-8346-9b081c3c0a3c deleted.


<Response [204]>